# Patrón Creacional: Abstract Factory

## Introducción
El patrón Abstract Factory permite producir familias de objetos relacionados sin especificar sus clases concretas. Es ideal cuando un sistema debe ser independiente de cómo se crean, componen y representan sus productos.

## Objetivos
- Comprender el propósito y la implementación del patrón Abstract Factory.
- Identificar cuándo es útil y cuándo evitarlo.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Banco**
Supón que tu banco ofrece productos para clientes personales y empresariales. Cada tipo de cliente necesita una familia de productos: tarjetas, cuentas y préstamos. Abstract Factory permite crear familias de productos consistentes para cada tipo de cliente sin acoplar el código a las clases concretas.

**¿Dónde se usa en proyectos reales?**
En sistemas de interfaces gráficas (diferentes temas visuales), aplicaciones bancarias (productos para distintos segmentos), sistemas de gestión de dispositivos, etc.

## Sin patrón Abstract Factory (forma errónea)
El código cliente debe conocer las clases concretas de cada producto. Esto hace difícil cambiar la familia de productos o agregar nuevas variantes.

In [1]:
class SillaModerna:
    def sentarse(self):
        print('Sentado en silla moderna')

class SillaVictoriana:
    def sentarse(self):
        print('Sentado en silla victoriana')

silla = SillaModerna()
silla.sentarse()

Sentado en silla moderna


## Con patrón Abstract Factory (forma correcta)
El cliente solo interactúa con la fábrica abstracta y no necesita saber los detalles de cada producto. Esto facilita la extensión y el mantenimiento.

In [2]:
class MueblesFactory:
    def crear_silla(self):
        raise NotImplementedError

class ModernaFactory(MueblesFactory):
    def crear_silla(self):
        return SillaModerna()

class VictorianaFactory(MueblesFactory):
    def crear_silla(self):
        return SillaVictoriana()

factory = VictorianaFactory()
silla = factory.crear_silla()
silla.sentarse()

Sentado en silla victoriana


## UML del patrón Abstract Factory
```plantuml
@startuml
abstract class MueblesFactory {
    + crear_silla()
}
class ModernaFactory
class VictorianaFactory
MueblesFactory <|-- ModernaFactory
MueblesFactory <|-- VictorianaFactory
class SillaModerna
class SillaVictoriana
ModernaFactory ..> SillaModerna
VictorianaFactory ..> SillaVictoriana
@enduml
```

## Otro ejemplo de la vida real: Aprovisionamiento de infraestructura multi-nube
**Contexto:** una plataforma SaaS despliega su infraestructura en más de un proveedor de nube (AWS o GCP) según el cliente o la región. Cada entorno necesita una **familia consistente de recursos**: cómputo (una instancia/VM) y almacenamiento (un bucket), y ambos deben pertenecer al **mismo proveedor** — mezclar cómputo de AWS con almacenamiento de GCP dispara latencia, egress de red y costos innecesarios. Este es exactamente el problema que Abstract Factory resuelve: garantizar que los productos de una familia sean siempre compatibles entre sí.

### Sin patrón (forma errónea)
Nada impide que el código combine, por error, componentes de proveedores distintos.

In [3]:
class InstanciaAWS:
    def desplegar(self):
        print('Desplegando instancia EC2 en AWS')

class BucketGCP:
    def guardar(self, archivo):
        print(f'Guardando "{archivo}" en Cloud Storage de GCP')

# El cliente arma el entorno "a mano" y termina mezclando proveedores sin darse cuenta
computo = InstanciaAWS()
almacenamiento = BucketGCP()  # bug: quedó mezclado con AWS

computo.desplegar()
almacenamiento.guardar('backup.zip')

Desplegando instancia EC2 en AWS
Guardando "backup.zip" en Cloud Storage de GCP


### Con patrón (forma correcta)
Una fábrica concreta por proveedor (`AWSFactory`, `GCPFactory`) es la única responsable de producir cómputo + almacenamiento; el cliente nunca puede combinar familias distintas porque solo trabaja con una fábrica a la vez.

In [4]:
import abc

class Computo(abc.ABC):
    @abc.abstractmethod
    def desplegar(self):
        ...

class Almacenamiento(abc.ABC):
    @abc.abstractmethod
    def guardar(self, archivo):
        ...


class InstanciaAWS(Computo):
    def desplegar(self):
        print('Desplegando instancia EC2 en AWS')

class BucketAWS(Almacenamiento):
    def guardar(self, archivo):
        print(f'Guardando "{archivo}" en S3 de AWS')

class InstanciaGCP(Computo):
    def desplegar(self):
        print('Desplegando VM de Compute Engine en GCP')

class BucketGCP(Almacenamiento):
    def guardar(self, archivo):
        print(f'Guardando "{archivo}" en Cloud Storage de GCP')


class ProveedorNubeFactory(abc.ABC):
    @abc.abstractmethod
    def crear_computo(self):
        ...
    @abc.abstractmethod
    def crear_almacenamiento(self):
        ...


class AWSFactory(ProveedorNubeFactory):
    def crear_computo(self):
        return InstanciaAWS()
    def crear_almacenamiento(self):
        return BucketAWS()


class GCPFactory(ProveedorNubeFactory):
    def crear_computo(self):
        return InstanciaGCP()
    def crear_almacenamiento(self):
        return BucketGCP()


def aprovisionar_entorno(factory: ProveedorNubeFactory):
    computo = factory.crear_computo()
    almacenamiento = factory.crear_almacenamiento()
    computo.desplegar()
    almacenamiento.guardar('backup.zip')

aprovisionar_entorno(AWSFactory())
aprovisionar_entorno(GCPFactory())

Desplegando instancia EC2 en AWS
Guardando "backup.zip" en S3 de AWS
Desplegando VM de Compute Engine en GCP
Guardando "backup.zip" en Cloud Storage de GCP


### UML del ejemplo multi-nube
```plantuml
@startuml
abstract class ProveedorNubeFactory {
    + crear_computo()
    + crear_almacenamiento()
}
class AWSFactory
class GCPFactory
ProveedorNubeFactory <|-- AWSFactory
ProveedorNubeFactory <|-- GCPFactory

interface Computo {
    + desplegar()
}
interface Almacenamiento {
    + guardar(archivo)
}
Computo <|.. InstanciaAWS
Computo <|.. InstanciaGCP
Almacenamiento <|.. BucketAWS
Almacenamiento <|.. BucketGCP

AWSFactory ..> InstanciaAWS
AWSFactory ..> BucketAWS
GCPFactory ..> InstanciaGCP
GCPFactory ..> BucketGCP
@enduml
```

### ¿Dónde más se usa Abstract Factory?
- **SDKs multi-nube / multi-proveedor:** Terraform o Pulumi seleccionan la familia de recursos (red, cómputo, almacenamiento) del proveedor configurado, sin que el resto del código sepa si es AWS, Azure o GCP.
- **Drivers de UI multiplataforma:** frameworks como Qt crean la familia de widgets (botón, checkbox, ventana) nativa de Windows, macOS o Linux.
- **E-commerce con kits de accesorios por marca:** una funda, un cargador y un protector de pantalla deben pertenecer a la misma familia (Apple o Samsung) para ser compatibles.
- **Motores de videojuegos:** una fábrica de "assets" por tema visual (medieval, futurista) que crea consistentemente personajes, armas y escenarios del mismo estilo.
- **Sistemas de pago regionales:** una fábrica que produce la familia de validador + formateador de moneda + pasarela correcta según el país, sin mezclar reglas de un país con la pasarela de otro.

**Ejercicio de reflexión:** ¿qué pasaría en el ejemplo multi-nube si en vez de 2 productos (cómputo, almacenamiento) tuvieras que agregar un tercero (red) a las dos familias existentes? ¿Cuántas clases tendrías que tocar con Abstract Factory, y cuántas sin él?

## Actividad
Crea tu propia Abstract Factory para una familia de productos tecnológicos (por ejemplo, Laptop y Smartphone de diferentes marcas).

---

## Explicación de conceptos clave
- **Consistencia:** Abstract Factory asegura que los productos de una familia sean compatibles entre sí.
- **Desacoplamiento:** El cliente no depende de las clases concretas de los productos.
- **Escalabilidad:** Es fácil agregar nuevas familias de productos sin modificar el código cliente.

## Conclusión
El patrón Abstract Factory es esencial cuando necesitas crear familias de objetos relacionados y garantizar su compatibilidad. Es muy útil en aplicaciones bancarias, sistemas de interfaces gráficas y cualquier sistema donde la variedad de productos pueda crecer o cambiar según el contexto del usuario.